<a href="https://colab.research.google.com/github/n1lays1ngh/Autoshorts-GreenAI/blob/main/VideoMAE_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade torchao


In [2]:
# Cell 1: Install Dependencies
!pip install -q transformers peft accelerate pandas opencv-python

import os
import cv2
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import zipfile
import shutil
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import VideoMAEModel
from peft import LoraConfig, get_peft_model
from torch.cuda.amp import autocast, GradScaler

# Verify GPU
!nvidia-smi

Mon Jul 13 09:05:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
# Let's see what folder is actually inside /content/dataset
print(os.listdir('/content/dataset'))

['autoshorts_training_data', '__MACOSX']


In [4]:
# Cell 2: Mount Drive and Extract to Local Scratch
from google.colab import drive
drive.mount('/content/drive')

drive_zip_path = '/content/drive/MyDrive/autoshorts_training_data.zip'
local_scratch_dir = '/content/dataset'

print("Copying zip file from Drive to local NVMe storage (this takes a minute)...")
os.makedirs(local_scratch_dir, exist_ok=True)
shutil.copy2(drive_zip_path, '/content/autoshorts_training_data.zip')

print("Extracting...")
with zipfile.ZipFile('/content/autoshorts_training_data.zip', 'r') as zip_ref:
    zip_ref.extractall(local_scratch_dir)

print(f"Extraction complete! Files available in {local_scratch_dir}")
# Verify paths
csv_path = os.path.join(local_scratch_dir, 'autoshorts_training_data', 'manifest_balanced.csv')
clips_dir = os.path.join(local_scratch_dir, 'autoshorts_training_data', 'clips')

print("CSV Exists:", os.path.exists(csv_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying zip file from Drive to local NVMe storage (this takes a minute)...
Extracting...
Extraction complete! Files available in /content/dataset
CSV Exists: True


In [9]:
# Cell 3: Dataset Class, Splitting, and DataLoader
import os
import cv2
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit

# Update these paths to step inside the Mac-generated zip folder
local_scratch_dir = '/content/dataset'
csv_path = os.path.join(local_scratch_dir, 'autoshorts_training_data', 'manifest_balanced.csv')
clips_dir = os.path.join(local_scratch_dir, 'autoshorts_training_data', 'clips')

class AutoShortsDataset(Dataset):
    def __init__(self, df, clips_dir, num_frames=16):
        self.df = df.reset_index(drop=True)
        self.clips_dir = clips_dir
        self.num_frames = num_frames

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = os.path.join(self.clips_dir, os.path.basename(row['clip_path']))

        # Target Saliency (0.0 to 1.0)
        label = torch.tensor(row['retention_val'], dtype=torch.float32)

        cap = cv2.VideoCapture(video_path)
        frames = []
        blank_frame = np.zeros((224, 224, 3), dtype=np.uint8)

        if not cap.isOpened():
            frames = [blank_frame for _ in range(self.num_frames)]
        else:
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            # Robust frame sampling fallback
            if total_frames > 0:
                indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
            else:
                indices = np.zeros(self.num_frames, dtype=int)

            for i in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, i)
                ret, frame = cap.read()
                if ret:
                    frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                else:
                    frames.append(blank_frame)
            cap.release()

        # Stack into shape: [Time, Channels, Height, Width] -> [16, 3, 224, 224]
        # This is EXACTLY the format Hugging Face VideoMAE expects natively.
        tensor_frames = torch.stack([self.transform(f) for f in frames])

        return tensor_frames, label

# Read full dataset
full_df = pd.read_csv(csv_path)

# CRITICAL FIX: Split by video_id to prevent data leakage!
# This ensures all clips from a specific video stay completely together
# in either the train OR the val set.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss.split(full_df, groups=full_df['video_id']))

train_df = full_df.iloc[train_idx].reset_index(drop=True)
val_df = full_df.iloc[val_idx].reset_index(drop=True)

train_dataset = AutoShortsDataset(train_df, clips_dir)
val_dataset = AutoShortsDataset(val_df, clips_dir)

# Initialize Dataloaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Unique videos in Train: {train_df['video_id'].nunique()}")
print(f"Unique videos in Val: {val_df['video_id'].nunique()}")

Train batches: 256 | Val batches: 61
Unique videos in Train: 267
Unique videos in Val: 67


In [6]:
# Cell 4: LoRA Model Fixed
class VideoMAELoRARegressor(nn.Module):
    def __init__(self, model_name="MCG-NJU/videomae-base"):
        super().__init__()
        self.base_model = VideoMAEModel.from_pretrained(model_name)

        # FIX: Correct ViT attention target modules
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["query", "value"],
            lora_dropout=0.1,
            bias="none"
        )

        self.encoder = get_peft_model(self.base_model, lora_config)
        self.encoder.print_trainable_parameters()

        self.regression_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, pixel_values):
        outputs = self.encoder(pixel_values)
        pooled_output = outputs.last_hidden_state.mean(dim=1)
        return self.regression_head(pooled_output)

# Set seed for reproducibility
torch.manual_seed(42)
model = VideoMAELoRARegressor().cuda()

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.norm.bias                                                    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED

trainable params: 294,912 || all params: 86,531,328 || trainable%: 0.3408


In [10]:
# Cell 5: Training Loop with Eval, Clipping, Schedulers, and Early Stopping
import torch.amp
from torch.optim.lr_scheduler import CosineAnnealingLR
import copy

EPOCHS, ACCUMULATION_STEPS, LEARNING_RATE = 10, 8, 1e-4
PATIENCE = 3 # Early stopping patience

# PRO FIX: Decouple weight decay for biases and LayerNorms
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and 'bias' not in n and 'LayerNorm' not in n]
no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and ('bias' in n or 'LayerNorm' in n)]

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': 1e-3},
    {'params': no_decay_params, 'weight_decay': 0.0}
], lr=LEARNING_RATE)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.HuberLoss()
scaler = torch.amp.GradScaler('cuda')

best_val_loss = float('inf')
patience_counter = 0
save_dir = '/content/drive/MyDrive/autoshorts_checkpoints'
os.makedirs(save_dir, exist_ok=True)

print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for i, (video_clips, target_scores) in enumerate(train_loader):
        video_clips, target_scores = video_clips.cuda(), target_scores.cuda()

        with torch.amp.autocast('cuda'):
            predictions = model(video_clips)
            loss = criterion(predictions.squeeze(), target_scores) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += (loss.item() * ACCUMULATION_STEPS)

    scheduler.step()

    # --- VALIDATION LOOP ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for video_clips, target_scores in val_loader:
            video_clips, target_scores = video_clips.cuda(), target_scores.cuda()
            with torch.amp.autocast('cuda'):
                preds = model(video_clips)
                val_loss += criterion(preds.squeeze(), target_scores).item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {epoch_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f}")

    # Checkpointing - Save Best Model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        print(f"--> New best val loss. Saving 'best' checkpoint...")
        model.encoder.save_pretrained(os.path.join(save_dir, 'best_lora_adapter'))
        torch.save(model.regression_head.state_dict(), os.path.join(save_dir, 'best_regression_head.pth'))
    else:
        patience_counter += 1
        print(f"--> No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered. Halting training.")
            break

# Final Save of the last state for comparison
print("Saving 'last_epoch' checkpoint...")
model.encoder.save_pretrained(os.path.join(save_dir, 'last_lora_adapter'))
torch.save(model.regression_head.state_dict(), os.path.join(save_dir, 'last_regression_head.pth'))
print("Training Complete. Models secured on Google Drive.")

Starting Training...
Epoch [1/10] | Train Loss: 0.0504 | Val Loss: 0.0487
--> New best val loss. Saving 'best' checkpoint...
Epoch [2/10] | Train Loss: 0.0491 | Val Loss: 0.0495
--> No improvement. Patience: 1/3
Epoch [3/10] | Train Loss: 0.0487 | Val Loss: 0.0491
--> No improvement. Patience: 2/3
Epoch [4/10] | Train Loss: 0.0481 | Val Loss: 0.0486
--> New best val loss. Saving 'best' checkpoint...
Epoch [5/10] | Train Loss: 0.0486 | Val Loss: 0.0486
--> New best val loss. Saving 'best' checkpoint...
Epoch [6/10] | Train Loss: 0.0483 | Val Loss: 0.0489
--> No improvement. Patience: 1/3
Epoch [7/10] | Train Loss: 0.0485 | Val Loss: 0.0486
--> No improvement. Patience: 2/3
Epoch [8/10] | Train Loss: 0.0483 | Val Loss: 0.0486
--> No improvement. Patience: 3/3
Early stopping triggered. Halting training.
Saving 'last_epoch' checkpoint...
Training Complete. Models secured on Google Drive.
